In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configuración
semanas = 25
dias_simulacion = semanas * 7
fecha_inicio = datetime(2023, 12, 31) # Empieza en Domingo
densidad_std = 1.030
nombre_fichero = 'queseria_final_tidy.csv'

dias_es = {
    'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles',
    'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}

data = []
stock_leche_litros = 0.0
stock_grasa_kg_total = 0.0
stock_proteina_kg_total = 0.0

for i in range(dias_simulacion):
    fecha = fecha_inicio + timedelta(days=i)
    dia_semana_es = dias_es[fecha.strftime('%A')]
    num_dia = fecha.weekday() 
    
    entrada_kg, entrada_litros, grasa_g_l, prot_g_l = 0, 0, 0.0, 0.0
    
    # 1. ENTRADAS (Domingo a Jueves)
    if num_dia in [6, 0, 1, 2, 3]:
        entrada_kg = int(round(np.random.normal(5150, 300))) 
        entrada_litros = int(round(entrada_kg / densidad_std))
        grasa_g_l = np.random.normal(37, 1.5)
        prot_g_l = np.random.normal(32, 1.0)
        
        stock_leche_litros += entrada_litros
        stock_grasa_kg_total += (entrada_litros * grasa_g_l) / 1000
        stock_proteina_kg_total += (entrada_litros * prot_g_l) / 1000

    # 2. FABRICACIÓN (Lunes a Viernes)
    envio_litros_final, envio_grasa_g_l, envio_prot_g_l = 0, 0.0, 0.0
    perdida_mg_dia, perdida_mp_dia, inc_agua_dia = 0.0, 0.0, 0.0
    
    if num_dia in [0, 1, 2, 3, 4]:
        # Lógica de envío teórico: 
        # Si es viernes, todo el stock. Si no, exactamente 5000.
        if num_dia == 4:
            envio_litros_teoricos = stock_leche_litros
        else:
            envio_litros_teoricos = 5000.0 if stock_leche_litros >= 5000 else stock_leche_litros

        if stock_leche_litros > 0 and envio_litros_teoricos > 0:
            factor = (envio_litros_teoricos / stock_leche_litros)
            kg_grasa_t = stock_grasa_kg_total * factor
            kg_prot_t = stock_proteina_kg_total * factor
            
            # Variables de proceso (Pérdidas y Agua)
            perdida_mg_dia = np.random.normal(0.04, 0.005)
            perdida_mp_dia = np.random.normal(0.04, 0.005)
            inc_agua_dia = np.random.normal(0.005, 0.001)
            
            # Resultado final en litros (enteros)
            envio_litros_final = int(round(envio_litros_teoricos * (1 + inc_agua_dia)))
            
            # Analítica final (g/L)
            envio_grasa_g_l = (kg_grasa_t * (1 - perdida_mg_dia) * 1000) / envio_litros_final
            envio_prot_g_l = (kg_prot_t * (1 - perdida_mp_dia) * 1000) / envio_litros_final
            
            # Actualización del stock físico de la quesería
            stock_leche_litros -= envio_litros_teoricos
            stock_grasa_kg_total -= kg_grasa_t
            stock_proteina_kg_total -= kg_prot_t

    # 3. GUARDAR DATOS
    data.append({
        'fecha': fecha.strftime('%d/%m/%Y'),
        'dia_semana': dia_semana_es,
        'semana': (i // 7) + 1,
        'entrada_kg': int(entrada_kg),
        'entrada_litros': int(entrada_litros),
        'entrada_grasa_g_l': round(grasa_g_l, 2),
        'entrada_prot_g_l': round(prot_g_l, 2),
        'envio_fab_litros': int(envio_litros_final),
        'envio_fab_grasa_g_l': round(envio_grasa_g_l, 2),
        'envio_fab_prot_g_l': round(envio_prot_g_l, 2),
        'ratio_perdida_mg': round(perdida_mg_dia, 5),
        'ratio_perdida_mp': round(perdida_mp_dia, 5),
        'ratio_inc_agua': round(inc_agua_dia, 5),
        'stock_cierre_litros': int(round(max(0, stock_leche_litros)))
    })

# Exportación con ISO-8859-1 para evitar problemas con acentos y columnas en Python
pd.DataFrame(data).to_csv(nombre_fichero, index=False, sep=';', decimal=',', encoding='ISO-8859-1')

print(f"Simulación generada. Fabricación fija de 5000L (Lun-Jue) y vaciado los Viernes.")

Simulación generada. Fabricación fija de 5000L (Lun-Jue) y vaciado los Viernes.
